# MuTap AEC vs. WebRTC AEC3 and Speex MDF

Plots the committed comparison results. Prose, methodology and the full
reading guide live in [`docs/aec-comparison.md`](../docs/aec-comparison.md).

Two directions:
1. **Their algorithms through our tests** — `bench/compare/results/baseline.json`
2. **Our algorithm through their test (AECMOS)** — `bench/compare/results/aecmos.json`

Needs only `numpy` and `matplotlib`; it reads the JSON the harness already wrote.

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt

root = os.path.join(os.path.dirname(os.getcwd()), 'bench', 'compare', 'results') \
       if os.path.basename(os.getcwd()) == 'notebooks' else 'bench/compare/results'
base = json.load(open(os.path.join(root, 'baseline.json')))['results']
print(f"{len(base)} rows:", sorted({r['subject'] for r in base}))

## Direction 1 — echo depth, convergence, near-end preservation, cost

One panel per rate. Bars are grouped by subject.

In [ ]:
def rows_at(fs): return [r for r in base if r['fs']==fs]
metrics = [('erle_db','ERLE dB (↑)'),('converge_ms','converge ms (↓)'),
           ('reconverge_ms','reconverge ms (↓)'),('dt_near_keep_db','DT near-keep dB (0=ideal)'),
           ('dt_echo_supp_db','DT echo dB (↑)'),('x_realtime','x realtime (↑)')]
for fs in [48000,16000]:
    rs = rows_at(fs); subs=[r['subject'] for r in rs]
    fig,axs=plt.subplots(2,3,figsize=(15,7)); fig.suptitle(f'Direction 1 — {fs//1000} kHz',fontsize=14)
    for ax,(k,lab) in zip(axs.flat,metrics):
        vals=[min(r[k],5000) if 'ms' in k else r[k] for r in rs]
        ax.bar(range(len(subs)),vals,color='tab:blue'); ax.set_title(lab,fontsize=10)
        ax.set_xticks(range(len(subs))); ax.set_xticklabels(subs,rotation=40,ha='right',fontsize=8)
        ax.grid(axis='y',alpha=0.3)
    plt.tight_layout(); plt.show()

### The trade in one plot: ERLE depth vs. convergence speed
MuTap buys steady-state depth; AEC3 buys convergence/tracking speed.

In [ ]:
rs=rows_at(48000)
fig,ax=plt.subplots(figsize=(8,5))
for r in rs:
    ax.scatter(r['converge_ms'],r['erle_db'],s=80)
    ax.annotate(r['subject'],(r['converge_ms'],r['erle_db']),xytext=(6,4),textcoords='offset points')
ax.set_xlabel('convergence to 20 dB (ms) — lower better')
ax.set_ylabel('steady ERLE (dB) — higher better')
ax.set_xscale('symlog'); ax.set_title('48 kHz: depth vs speed'); ax.grid(alpha=0.3); plt.show()

## Direction 2 — AECMOS (their metric)

Echo MOS (higher = less audible echo) per scenario. See the data caveat in
the doc: the committed run is a synthetic eval set, so read **relative**
ordering, not absolute MOS.

In [ ]:
ap=os.path.join(root,'aecmos.json')
if os.path.exists(ap):
    am=json.load(open(ap)); rows=am['rows']
    subs=sorted({r['subject'] for r in rows}); scen=['st','dt','nst']
    fig,ax=plt.subplots(figsize=(9,5)); w=0.25
    for i,s in enumerate(scen):
        vals=[next((r['echo_mos'] for r in rows if r['subject']==u and r['scenario']==s),0) for u in subs]
        ax.bar(np.arange(len(subs))+i*w,vals,w,label=f'{s} echoMOS')
    ax.set_xticks(np.arange(len(subs))+w); ax.set_xticklabels(subs)
    ax.set_ylabel('AECMOS echo MOS (↑)'); ax.set_ylim(3.5,5.05)
    ax.set_title(f"AECMOS ({'synthetic — relative only' if am['synthetic'] else 'real corpus'})")
    ax.legend(); ax.grid(axis='y',alpha=0.3); plt.show()
else:
    print('run tools/compare/aecmos_eval.py first')